# CareLoop Clinics — No-Show Guard
## Machine Learning: Predicting Appointment No-Shows & Decision Threshold Calibration

**Author:** Tarun Shahi  
**Project:** CareLoop Clinics Patient No-Show Analytics & Early Risk Prediction  
**Input Data:** `appointments_clean.csv`  
**Deliverable:** `test_risk_scores.csv`

---

### Operational Context
The prediction point is the **evening prior to the scheduled appointment**, immediately after that day's automated reminder notifications have been sent or skipped. At this point:
- All feature values (`lead_days`, `distance_km`, `fee_inr`, `prior_no_shows`, `reminder_sent`, `booking_channel`, `visit_type`) are known.
- The front-desk team can make targeted telephone calls to patients flagged as high risk.

### Methodology & Assumptions
1. **Holdout Test Set:** The test set (20%, N=198) is reserved strictly for final reporting. Model selection, hyperparameter tuning, and threshold calibration are performed solely on the training set (80%, N=792) using 5-fold cross-validation.
2. **Preprocessing Boundary:** In Task 1.3, global imputation and outlier fences were calculated across the full cleaned dataset; in production, these are fitted within training folds.
3. **Observational Data:** Feature effects indicate statistical associations rather than proven causal relationships.


## 1. Feature Preparation & Train/Test Split

Prepare features observable at the evening prediction point:
- Numeric: `age`, `distance_km`, `fee_inr`, `lead_days`, `prior_no_shows`.
- Binary: `reminder_sent` (mapped to 1 for 'Yes', 0 for 'No').
- Categorical: `booking_channel` and `visit_type` (one-hot encoded).

Target variable: `y = df["no_show"]`. Partition into an 80/20 train/test split with `random_state=42`.


In [1]:
import os
from pathlib import Path

def resolve_path(path_str: str) -> str:
    for candidate in [
        Path(path_str),
        Path("..") / path_str,
        Path("../processed") / Path(path_str).name,
        Path("processed") / Path(path_str).name,
        Path("../data/processed") / Path(path_str).name,
        Path("data/processed") / Path(path_str).name
    ]:
        if candidate.exists():
            return str(candidate)
    return path_str

PROCESSED_DIR = Path(".")
# Output directory set to repository root
CLEAN_APPTS_PATH = resolve_path("appointments_clean.csv")

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Load clean appointments dataset without re-sorting or shuffling
df = pd.read_csv(CLEAN_APPTS_PATH)

# Feature selection
feature_cols = [
    "age", "distance_km", "fee_inr", "lead_days", "prior_no_shows",
    "reminder_sent", "booking_channel", "visit_type"
]

X_raw = df[feature_cols].copy()
# Encode reminder_sent as binary integer
X_raw["reminder_sent"] = X_raw["reminder_sent"].map({"Yes": 1, "No": 0})

# One-hot encode categorical features
X = pd.get_dummies(X_raw, columns=["booking_channel", "visit_type"], dtype=int)
y = df["no_show"]

print(f"Total features in X ({X.shape[1]} columns):")
print(list(X.columns))

# Split 80/20 with random_state=42
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

n_train = len(X_train)
n_test = len(X_test)
test_no_shows = y_test.sum()
test_no_show_rate = test_no_shows / n_test
baseline_accuracy = (y_test == 0).sum() / n_test

print(f"\nTraining set size: {n_train} rows")
print(f"Test set size:     {n_test} rows")
print(f"Test set no-shows: {test_no_shows} ({test_no_show_rate*100:.1f}%)")
print(f"Baseline accuracy (always predicting 'Attended'): {baseline_accuracy:.4f} ({(y_test == 0).sum()}/{n_test})")


Total features in X (11 columns):
['age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows', 'reminder_sent', 'booking_channel_App', 'booking_channel_Phone', 'booking_channel_Walk-In', 'visit_type_Consultation', 'visit_type_Follow-up']

Training set size: 792 rows
Test set size:     198 rows
Test set no-shows: 49 (24.7%)
Baseline accuracy (always predicting 'Attended'): 0.7525 (149/198)


## 2. Logistic Regression Baseline & Feature Effects

Fit a Logistic Regression model with features standardized using `StandardScaler` fitted on `X_train`.

Evaluate test accuracy, confusion matrix, precision, recall, F1-score, and ROC-AUC. Examine standardized coefficients and odds ratios to understand feature associations (particularly for appointment reminders).


In [2]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, precision_score,
    recall_score, f1_score, roc_auc_score
)

# Standardize features (fit on train only)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# Fit Logistic Regression
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_s, y_train)

# Evaluate on test set (threshold = 0.5)
y_pred_lr = lr_model.predict(X_test_s)
y_prob_lr = lr_model.predict_proba(X_test_s)[:, 1]

acc_lr = accuracy_score(y_test, y_pred_lr)
cm_lr = confusion_matrix(y_test, y_pred_lr)
prec_lr = precision_score(y_test, y_pred_lr)
rec_lr = recall_score(y_test, y_pred_lr)
f1_lr = f1_score(y_test, y_pred_lr)
auc_lr = roc_auc_score(y_test, y_prob_lr)

print("--- Task 3.2 Logistic Regression Test Metrics (Threshold 0.5) ---")
print(f"Accuracy:         {acc_lr:.4f}")
print(f"Confusion Matrix: \n{cm_lr}")
print(f"Precision:        {prec_lr:.4f}")
print(f"Recall:           {rec_lr:.4f}")
print(f"F1-score:         {f1_lr:.4f}")
print(f"ROC-AUC:          {auc_lr:.4f}")

# Feature coefficients
coef_series = pd.Series(lr_model.coef_[0], index=X.columns)
print("\nModel Coefficients (Standardized):")
for feat, c in coef_series.items():
    print(f"  {feat:25s}: {c:+.4f}")

# Reminder effect analysis
rem_idx = list(X.columns).index("reminder_sent")
beta_rem = lr_model.coef_[0][rem_idx]
sd_rem = scaler.scale_[rem_idx]
or_per_sd = np.exp(beta_rem)
or_unit_change = np.exp(beta_rem / sd_rem)

print(f"\nreminder_sent coefficient (beta): {beta_rem:.4f}")
print(f"Odds multiplier per 1 SD (np.exp(beta)): {or_per_sd:.4f} (SD = {sd_rem:.4f})")
print(f"Odds ratio for No -> Yes (np.exp(beta / SD)): {or_unit_change:.4f}")
print(f"\nConclusion: In this fitted model, holding all other features constant, sending an appointment reminder")
print(f"is associated with an estimated {(1 - or_unit_change)*100:.1f}% reduction in the odds of a patient no-show")
print(f"(Odds Ratio ≈ {or_unit_change:.2f}) — an observational association, not proof of causality.")


--- Task 3.2 Logistic Regression Test Metrics (Threshold 0.5) ---
Accuracy:         0.7879
Confusion Matrix: 
[[139  10]
 [ 32  17]]
Precision:        0.6296
Recall:           0.3469
F1-score:         0.4474
ROC-AUC:          0.7889

Model Coefficients (Standardized):
  age                      : -0.2539
  distance_km              : +0.2534
  fee_inr                  : +0.4528
  lead_days                : +0.6145
  prior_no_shows           : +0.6496
  reminder_sent            : -0.5807
  booking_channel_App      : -0.0468
  booking_channel_Phone    : +0.2159
  booking_channel_Walk-In  : -0.1864
  visit_type_Consultation  : +0.0157
  visit_type_Follow-up     : -0.0157

reminder_sent coefficient (beta): -0.5807
Odds multiplier per 1 SD (np.exp(beta)): 0.5595 (SD = 0.4950)
Odds ratio for No -> Yes (np.exp(beta / SD)): 0.3094

Conclusion: In this fitted model, holding all other features constant, sending an appointment reminder
is associated with an estimated 69.1% reduction in the odds of

## 3. Decision Tree Benchmark & Overfitting Diagnostic

Train two decision trees (`DecisionTreeClassifier(random_state=42)`):
1. **Unconstrained Tree (full depth):** Evaluates training memorization and overfitting.
2. **Constrained Tree (`max_depth=3`):** Applies shallow pruning to test generalization.


In [3]:
from sklearn.tree import DecisionTreeClassifier

# (1) Full-depth unconstrained tree
dt_full = DecisionTreeClassifier(random_state=42)
dt_full.fit(X_train, y_train)

full_train_acc = accuracy_score(y_train, dt_full.predict(X_train))
full_test_acc = accuracy_score(y_test, dt_full.predict(X_test))
full_depth = dt_full.get_depth()

print(f"Full-Depth Tree (depth={full_depth}):")
print(f"  Training Accuracy: {full_train_acc:.4f} (100.0%)")
print(f"  Test Accuracy:     {full_test_acc:.4f} ({full_test_acc*100:.1f}%)")

# (2) Constrained depth-3 tree
dt_d3 = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_d3.fit(X_train, y_train)

d3_train_acc = accuracy_score(y_train, dt_d3.predict(X_train))
d3_test_acc = accuracy_score(y_test, dt_d3.predict(X_test))
d3_test_rec = recall_score(y_test, dt_d3.predict(X_test))
d3_test_auc = roc_auc_score(y_test, dt_d3.predict_proba(X_test)[:, 1])

print(f"\nDepth-3 Constrained Tree:")
print(f"  Training Accuracy: {d3_train_acc:.4f}")
print(f"  Test Accuracy:     {d3_test_acc:.4f}")
print(f"  Test Recall:       {d3_test_rec:.4f}")
print(f"  Test ROC-AUC:      {d3_test_auc:.4f}")

print(f"\nOverfitting Diagnostic: The massive 32.8 percentage point collapse between training accuracy (1.0000)")
print(f"and test accuracy ({full_test_acc:.4f}) in the unconstrained tree demonstrates that it memorized idiosyncratic")
print(f"training noise (falling well below the naive baseline of {baseline_accuracy:.4f}), whereas constraining max_depth=3")
print(f"generalizes substantially better ({d3_test_acc:.4f} test accuracy).")


Full-Depth Tree (depth=15):
  Training Accuracy: 1.0000 (100.0%)
  Test Accuracy:     0.6717 (67.2%)

Depth-3 Constrained Tree:
  Training Accuracy: 0.8220
  Test Accuracy:     0.7778
  Test Recall:       0.2245
  Test ROC-AUC:      0.6997

Overfitting Diagnostic: The massive 32.8 percentage point collapse between training accuracy (1.0000)
and test accuracy (0.6717) in the unconstrained tree demonstrates that it memorized idiosyncratic
training noise (falling well below the naive baseline of 0.7525), whereas constraining max_depth=3
generalizes substantially better (0.7778 test accuracy).


## 4. Random Forest Ensemble & Baseline Comparison

Fit a `RandomForestClassifier(n_estimators=100, random_state=42)` and extract Gini feature importances. Compare test set performance against Logistic Regression and the shallow Decision Tree.


In [4]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

rf_train_acc = accuracy_score(y_train, rf_model.predict(X_train))
rf_test_acc = accuracy_score(y_test, rf_model.predict(X_test))
rf_test_rec = recall_score(y_test, rf_model.predict(X_test))
rf_test_auc = roc_auc_score(y_test, rf_model.predict_proba(X_test)[:, 1])

print("--- Task 3.4 Random Forest Test Metrics ---")
print(f"Training Accuracy: {rf_train_acc:.4f}")
print(f"Test Accuracy:     {rf_test_acc:.4f}")
print(f"Test Recall:       {rf_test_rec:.4f}")
print(f"Test ROC-AUC:      {rf_test_auc:.4f}")

# Top 3 feature importances
importances = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False)
print(f"\nTop 3 Feature Importances in Random Forest:")
for feat, imp in importances.head(3).items():
    print(f"  {feat:15s}: {imp:.4f} ({imp*100:.1f}%)")

# Comparison table (Reporting only)
comparison_df = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "Accuracy": round(acc_lr, 3),
        "Recall": round(rec_lr, 3),
        "ROC-AUC": round(auc_lr, 3)
    },
    {
        "Model": "Decision Tree (depth=3)",
        "Accuracy": round(d3_test_acc, 3),
        "Recall": round(d3_test_rec, 3),
        "ROC-AUC": round(d3_test_auc, 3)
    },
    {
        "Model": "Random Forest",
        "Accuracy": round(rf_test_acc, 3),
        "Recall": round(rf_test_rec, 3),
        "ROC-AUC": round(rf_test_auc, 3)
    }
])

print("\nModel Comparison Table (Holdout Test Set Reporting):")
print(comparison_df.to_string(index=False))

print("\nObservation: Logistic Regression delivers the strongest discrimination (ROC-AUC 0.789) and highest recall (0.347)")
print("among untuned models. Note: Final model selection is conducted strictly on training data in Task 3.5.")


--- Task 3.4 Random Forest Test Metrics ---
Training Accuracy: 1.0000
Test Accuracy:     0.7475
Test Recall:       0.3061
Test ROC-AUC:      0.7057

Top 3 Feature Importances in Random Forest:
  lead_days      : 0.2199 (22.0%)
  age            : 0.1968 (19.7%)
  distance_km    : 0.1947 (19.5%)

Model Comparison Table (Holdout Test Set Reporting):
                  Model  Accuracy  Recall  ROC-AUC
    Logistic Regression     0.788   0.347    0.789
Decision Tree (depth=3)     0.778   0.224    0.700
          Random Forest     0.747   0.306    0.706

Observation: Logistic Regression delivers the strongest discrimination (ROC-AUC 0.789) and highest recall (0.347)
among untuned models. Note: Final model selection is conducted strictly on training data in Task 3.5.


## 5. Model Selection (5-Fold CV), Threshold Tuning & Evaluation

1. **Hyperparameter Tuning:** Tune `RandomForestClassifier` on `X_train` with `GridSearchCV(cv=5, scoring="roc_auc")` across tree depth and estimators.
2. **Cross-Validation Comparison:** Compare 5-fold cross-validated ROC-AUC across Logistic Regression, Depth-3 Tree, Random Forest, and Tuned Forest.
3. **Threshold Calibration:** Evaluate out-of-fold training probabilities at thresholds 0.50 and 0.35 to balance precision and recall.
4. **Holdout Evaluation:** Evaluate the chosen model once on the holdout test set at the selected threshold, and export predictions to `test_risk_scores.csv`.


In [5]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, cross_val_score, cross_val_predict

# (a) Hyperparameter Tuning on Training Set
param_grid = {"n_estimators": [100, 200], "max_depth": [3, 5, 8]}
grid_rf = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring="roc_auc"
)
grid_rf.fit(X_train, y_train)

print(f"GridSearchCV Best Parameters: {grid_rf.best_params_}")
print(f"GridSearchCV Best Training CV ROC-AUC: {grid_rf.best_score_:.4f}")

# (b) Honest Model Selection via 5-Fold Training CV
# Pipeline ensures scaler is fitted strictly on fold training data without validation leakage
lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

cv_auc_lr = cross_val_score(lr_pipe, X_train, y_train, cv=5, scoring="roc_auc").mean()
cv_auc_dt3 = cross_val_score(DecisionTreeClassifier(max_depth=3, random_state=42), X_train, y_train, cv=5, scoring="roc_auc").mean()
cv_auc_rf = cross_val_score(RandomForestClassifier(n_estimators=100, random_state=42), X_train, y_train, cv=5, scoring="roc_auc").mean()
cv_auc_tuned_rf = grid_rf.best_score_

print("\n--- 5-Fold Cross-Validated Training ROC-AUC Benchmark ---")
print(f"  Logistic Regression Pipeline: {cv_auc_lr:.4f}  <-- WINNER")
print(f"  Decision Tree (depth=3):      {cv_auc_dt3:.4f}")
print(f"  Random Forest (default):     {cv_auc_rf:.4f}")
print(f"  Tuned Random Forest:         {cv_auc_tuned_rf:.4f}")

# (c) Out-of-Fold Threshold Selection on Training Set
oof_probs = cross_val_predict(lr_pipe, X_train, y_train, cv=5, method="predict_proba")[:, 1]

print("\n--- Out-of-Fold Threshold Evaluation on Training Data ---")
for threshold in [0.50, 0.35]:
    flagged = (oof_probs >= threshold).astype(int)
    p = precision_score(y_train, flagged)
    r = recall_score(y_train, flagged)
    f = f1_score(y_train, flagged)
    n = flagged.sum()
    print(f"Threshold {threshold:4.2f} -> Precision: {p:.4f}, Recall: {r:.4f}, F1: {f:.4f}, Flagged: {n:3d} patients")

print("\nThreshold Choice: Selected 0.35 because F1-score is substantially higher (0.489 vs 0.347) and recall nearly doubles")
print("(46.8% vs 24.8%). In outpatient clinic operations, failing to flag a true no-show leaves an unfillable empty doctor slot,")
print("whereas an extra outreach call costs only approximately one minute of call-centre staff time.")

# (d) Final Test Evaluation of Winner (Trained on full training set)
lr_pipe.fit(X_train, y_train)
final_test_probs = lr_pipe.predict_proba(X_test)[:, 1]
final_test_flags = (final_test_probs >= 0.35).astype(int)

final_precision = precision_score(y_test, final_test_flags)
final_recall = recall_score(y_test, final_test_flags)
final_f1 = f1_score(y_test, final_test_flags)
final_roc_auc = roc_auc_score(y_test, final_test_probs)
final_flagged_count = final_test_flags.sum()

print("\n--- Final Test Set Evaluation (Logistic Regression @ Threshold 0.35) ---")
print(f"Precision:        {final_precision:.4f} (57.1%)")
print(f"Recall:           {final_recall:.4f} (57.1%)")
print(f"F1-Score:         {final_f1:.4f}")
print(f"ROC-AUC:          {final_roc_auc:.4f}")
print(f"Flagged Patients: {final_flagged_count} of {len(y_test)} ({final_flagged_count/len(y_test)*100:.1f}%)")

# (e) Save test_risk_scores.csv
test_scores_df = pd.DataFrame({
    "appointment_id": df.loc[X_test.index, "appointment_id"].values,
    "no_show": y_test.values,
    "risk_prob": final_test_probs.round(4),
    "flag": final_test_flags
})
test_scores_df.to_csv("test_risk_scores.csv", index=False)
print(f"\nSaved test_risk_scores.csv ({len(test_scores_df)} rows, {test_scores_df['flag'].sum()} flagged)")

# (f) Operational Deployment Rule
print("\nOperational Deployment Principle:")
print("In real-world deployment, CareLoop would calibrate the risk threshold based on each clinic front desk's")
print("evening telephone call capacity balanced against the financial and operational cost of an unfilled doctor slot.")


GridSearchCV Best Parameters: {'max_depth': 3, 'n_estimators': 200}
GridSearchCV Best Training CV ROC-AUC: 0.7673



--- 5-Fold Cross-Validated Training ROC-AUC Benchmark ---
  Logistic Regression Pipeline: 0.7737  <-- WINNER
  Decision Tree (depth=3):      0.6784
  Random Forest (default):     0.7466
  Tuned Random Forest:         0.7673

--- Out-of-Fold Threshold Evaluation on Training Data ---
Threshold 0.50 -> Precision: 0.5753, Recall: 0.2485, F1: 0.3471, Flagged:  73 patients
Threshold 0.35 -> Precision: 0.5130, Recall: 0.4675, F1: 0.4892, Flagged: 154 patients

Threshold Choice: Selected 0.35 because F1-score is substantially higher (0.489 vs 0.347) and recall nearly doubles
(46.8% vs 24.8%). In outpatient clinic operations, failing to flag a true no-show leaves an unfillable empty doctor slot,
whereas an extra outreach call costs only approximately one minute of call-centre staff time.

--- Final Test Set Evaluation (Logistic Regression @ Threshold 0.35) ---
Precision:        0.5714 (57.1%)
Recall:           0.5714 (57.1%)
F1-Score:         0.5714
ROC-AUC:          0.7889
Flagged Patients: 4


Saved test_risk_scores.csv (198 rows, 49 flagged)

Operational Deployment Principle:
In real-world deployment, CareLoop would calibrate the risk threshold based on each clinic front desk's
evening telephone call capacity balanced against the financial and operational cost of an unfilled doctor slot.
